# Reproduce — Retained Spectral (credibility audit)

One click reproduces the whole benchmark on a fresh machine and **certifies its
credibility from the numbers measured here** — nothing is copied from the repo's
committed results.

Two axes are kept strictly separate (the project's tier-honesty rule):

1. **Credibility** — is the reproduction sound? Correctness at the *declared* tolerance on
   an 8-case adversarial suite (wells translated far off-origin, narrow/broad wells, an
   8-mode Morse well, a factorized double well, quartic scale covariance over 5 decades),
   a 20-point mode×grid scaling sweep cross-checked against SciPy, a recorded source commit,
   and a frozen single-thread environment. **This is what the audit ACCEPTs.**
2. **Speed / fairness** — does the native **Retained Multilevel Sturm** solver beat every
   standard eigensolver? Reported as `baseline_benchmark_verdict`, and it may honestly be
   **HOLD** when a competitor (e.g. ARPACK) fails to converge on a well — that never lowers
   the reproduction's credibility, and a green audit never implies universal dominance.

Tier: `finite_diagnostic` — a discrete rational-readout agreement-and-cost claim, not a
continuum-limit proof or an empirical-physics claim.


## 1 · Frozen environment, pinned to an immutable commit

The package is installed **from one exact audited commit** (not a floating `main`), so the
code cannot change between runs, and the pinned numeric stack makes the arithmetic
bit-reproducible. The commit is exported as `GITHUB_SHA` so every audit JSON records it and
the `source_commit_recorded` credibility gate passes. Single-thread BLAS is set before numpy
is imported anywhere.


In [ ]:
import os

# the EXACT audited source commit — reproducible bit-for-bit on any day, and recorded in every
# audit JSON so the credibility gate (source_commit_recorded) passes on a normal Colab run.
AUDITED_COMMIT = "5035a612facd5904e3c639a56edff6b72232e7a3"
os.environ["GITHUB_SHA"] = AUDITED_COMMIT

# single-thread BLAS/LAPACK before numpy is imported — measure the algorithm, not the scheduler
for _v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[_v] = '1'

# pinned numeric stack (== requirements-spectral-lock.txt), then the package AT THE AUDITED COMMIT
!pip -q install numpy==1.26.4 scipy==1.13.1 numba==0.60.0 llvmlite==0.43.0 matplotlib==3.9.2
!pip -q install "information-discrete-math[spectral-bench] @ git+https://github.com/morrocwi/information-discrete-math@{AUDITED_COMMIT}"

print("audited source commit:", AUDITED_COMMIT)

## 2 · Credibility audit (measured live)

Runs the consolidated `credibility_audit`: the baseline competition (**101 samples** for a
stable bootstrap CI) + 8 adversarial cases + a 20-point scaling sweep + cold-start, all
against the same **independent, reference-blind** SciPy pipeline (its well search is unbounded
— it is not handed the native window). The same-operator executor audit uses **31 repeats**.
ACCEPT here means the reproduction is correct and reproducible; the speed/fairness contest is
reported separately.

Correctness is checked THREE independent ways per spectrum (analytic reference, an
extended-precision recomputation of the identical operator, and a Sturm sign-count
certificate) — the `three_layer_correctness_all` gate.


In [ ]:
import json
from pathlib import Path
from retained_spectral.competition.credibility_audit import run_credibility_audit

# baseline_repeats=101 → the headline bootstrap CI is stable; executor_repeats=31 → same-operator
# audit CIs are stable. (A few minutes on a Colab CPU.)
audit = run_credibility_audit(include_jax=False, baseline_repeats=101,
                              executor_repeats=31, scaling_repeats=7)
Path('credibility-audit.json').write_text(json.dumps(audit, indent=2))

print('credibility_gates:', json.dumps(audit['credibility_gates'], indent=2))
print('credibility verdict           :', audit['verdict'])
print('baseline benchmark verdict    :', audit['baseline_benchmark_verdict'], '(reported, not a credibility gate)')
print('adversarial cases all correct :', audit['adversarial']['all_ok'], f"({len(audit['adversarial']['cases'])} cases)")
print('scaling cross-checks all ok   :', audit['scaling']['all_cross_checks'], f"({len(audit['scaling']['cases'])} points)")
print('source commit recorded        :', audit['baseline']['source_commit'])

## 3 · Charts (redrawn from the audited baseline)

The headline competition already ran inside the audit above (101 samples); the charts are
drawn from that exact record — no second run. Two figures:

* **hero** — median solve time, ranked by the *measured* time (the fastest bar gets the ★, so
  it moves to whichever solver actually won on this host), with a 25–75% spread whisker per bar;
* **detail** — a forest plot of the per-case speedup with its **95% bootstrap confidence
  interval** (colour = the recorded native_faster / tie / competitor_faster verdict, parity at
  1×), beside the **raw per-call samples** for native vs the independent SciPy pipeline.


In [ ]:
from pathlib import Path
from retained_spectral.competition.chart import render_hero, render_detail

baseline = audit['baseline']            # the 101-sample competition already measured above
print(json.dumps(baseline['end_to_end']['summary'], indent=2))
print('verdict_gates:', json.dumps(baseline['verdict_gates'], indent=2))
print('overall benchmark verdict:', baseline['verdict'], '| seed:', baseline['end_to_end']['seed'])

render_hero(baseline, Path('hero.png'))
render_detail(baseline, Path('detail.png'))
from IPython.display import Image, display
display(Image('hero.png'))
display(Image('detail.png'))

## 4 · Solve your own problem from raw input


In [ ]:
import retained_spectral as rs
problem = rs.make_problem(name='my', family='harmonic',
                          parameters={'omega': 2.0, 'center': 0.0}, modes=4)
r = rs.solve(problem)
print(r.status, r.values)   # ACCEPT (1.0, 3.0, 5.0, 7.0)